In [21]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from functools import reduce


In [28]:
fajlovi = glob.glob("/home/aleksandar/Desktop/kursVlada_dsml/CAPSTONE_PROJECT/4github/data/SCADA/2025/tbl*2025_01.csv")

In [ ]:
for fajl in fajlovi:
    print(fajl.split('/')[-1])
    

tblSCTurTemp_2025_01.csv
tblAlarmLog_2025_01.csv
tblSCTurIntern_2025_01.csv
tblDailySummary_2025_01.csv
tblSCTurbine_2025_01.csv
tblSCTurPress_2025_01.csv
tblSCTurDigiIn_2025_01.csv
tblSCTurCount_2025_01.csv
tblSCTurDigiOut_2025_01.csv
tblGrid_2025_01.csv
tblSCTurFlag_2025_01.csv
tblGridScientific_2025_01.csv
tblSCTurGrid_2025_01.csv


In [ ]:
DATA_DIR = Path("../data/2025/")              
OUT_FILE = Path("../data/SCADA/columns_2025_01.txt") # this file contains every .csv with columns

files = sorted(DATA_DIR.rglob("*_01.csv"))    

with open(OUT_FILE, "w", encoding="utf-8") as f:
    for file in files:
        cols = pd.read_csv(file, nrows=0).columns    
        f.write(f"{file.name} ({len(cols)} columns)\n")
        f.write("\n".join(cols) + "\n")
        f.write("-" * 14 + "\n")


****

## SCADA column selection

To see what the SCADA dataset contains, I listed the columns of all 14 tables for one month (January 2025) and saved them to `columns_2025_01.txt`. I went through the list and kept only what I need for an **hourly, farm-level dataset for April 2024 – April 2026**. The dataset needs three things: the **target** (energy at the metering point), **filters** (availability, stops, curtailment, icing) and **measured weather** for analysis.

The inputs of my model will be the Open-Meteo day-ahead forecasts, not SCADA data. Measured wind, direction and temperature are not known one day ahead, so I use them only to analyse the data, clean it and build the power curve.

In the end I kept 7 of the 14 tables and about 30 columns.

### What I kept

**Target.** I found two candidates for farm output: `ActivePowerExport` in `tblGrid` and `ActiveEnergy` / `ActivePowerMean` in `tblGridScientific`, which also has its own quality flag `DataOk`. I prefer the export at the metering point, because that is the energy that is nominated and settled, including electrical losses. I also kept `ActivePower` (`tblGrid`) and the power of each turbine (`wtc_ActPower_mean`). With them I can check the target against the sum of all turbines, and the turbine sum can serve as a backup target if the grid data has gaps.

**Measured weather (analysis only).**
- `wtc_AcWindSp_mean` – the main measured wind speed. I will use it for the power curve, as the target of the wind correction and for the "perfect wind" scenario.
- `wtc_AcWindSp_stddev` – for turbulence intensity (TI = std / mean), which changes the shape of the power curve.
- `wtc_ActualWindDirection_mean` and `wtc_NacelPos_mean` – wind direction for wake losses by sector; nacelle position for the northing correction.
- `wtc_AmbieTmp_mean` – for air density and icing conditions.

**Availability and stops.** `tblSCTurFlag` turned out to be the most useful table for filtering. It gives the seconds per 10-min interval that each turbine spent in each state:
- I count a turbine as available when it is in operation (`ScInOper`) or *ready to operate* (`ScReToOp`). A turbine that is ready is waiting for wind, so if I counted only `ScInOper`, I would remove all low-wind hours and bias the model.
- Stops caused by a turbine fault, the grid or a command (`ScTurSto`, `ScGrdSto`, `ScComSto`) have nothing to do with the weather, so I will filter them out.
- I kept environmental stops (`ScEnvSto`: high-wind cut-out, ice, cold) separately. They are weather-driven and really reduce production, so I am not sure yet whether the model should learn them or whether I should filter them. I will decide at the modelling stage.
- I also kept `wtc_ActPower_min`: a value near 0 while the mean is above 0 shows a stop inside the interval.

**Curtailment and regulation.** To recognise periods when the turbines were limited, I kept `wtc_PowerRef_endvalue` (power limit), `wtc_ActRegSt_endvalue` (regulation state) and `wtc_PowerRed_timeon` (time with power reduction active). I also kept `wtc_PitcPosA_mean`, because a high pitch angle at below-rated wind confirms curtailment. I kept only blade A, as B and C move together with it. `wtc_BoostAct_mean` shows when Power Boost was on, which changes the power curve. `wtc_WTOperSt_endvalue` and `WPSStatus` give the operating state of each turbine and of the whole farm.

**Data quality.** `wtc_ActPower_Quality_endvalue` flags bad power measurements. The primary and secondary anemometers (`wtc_PrWindSp_mean`, `wtc_SeWindSp_mean`) help me spot a failed or iced sensor. In `tblSCTurDigiIn` I found `wtc_IceRelay_timeon`, a direct ice detection signal, so I don't have to guess icing only from temperature and the power curve.

### What I left out

I dropped whole tables that are not related to energy production:
- `tblDailySummary` – daily values, too coarse for an hourly forecast.
- `tblSCTurCount` – cumulative counters; `tblSCTurFlag` gives the same information per interval.
- `tblSCTurDigiOut`, `tblSCTurIntern`, `tblSCTurPress` and most of `tblSCTurTemp` – pumps, heaters, pressures, controller internals and component temperatures. These are for condition monitoring, not for forecasting.
- `tblAlarmLog` – I may come back to it if I need to explain single stops, but the stop categories are already in `tblSCTurFlag`.

From the tables I kept, I left out:
- `min` / `max` / `stddev` of most signals, because the mean is enough. The exceptions are wind `stddev` (turbulence) and power `min` (stops).
- `wtc_PriAnemo_*` and `wtc_SecAnemo_*`, which look like raw signals of the same anemometers as `PrWindSp` / `SeWindSp`.
- Pitch setpoints (`wtc_PitchRef_*`, `wtc_PitcRefA_*`) and blades B and C, which carry the same information as `PitcPosA`.
- `wtc_YawPos_*` and `wtc_ScYawPos_*`, because nacelle orientation is already covered by `NacelPos`.
- Drivetrain and tower diagnostics (`GenRpm`, `MainSRpm`, tower frequency).
- `wtc_TwrHumid_mean`, because it is humidity inside the tower, not outside.
- `wtc_RawPower_*` (I use the standard `ActPower` signal) and `wtc_BoostAva_*` (only *active* boost matters, not *available*).
- Voltage, current, reactive power, frequency and power factor, which describe electrical quality, not energy.

### Next steps
1. **Choose the target.** Check which grid candidate is a per-interval value and which is a cumulative counter (a counter only grows, so I would need `.diff()`), check the units and compare both with the sum of turbines. Then keep one.
2. **Check the state columns.** Their meaning so far comes only from the names, so I will look them up in the data dictionary and run `value_counts()` before setting any thresholds.
3. **Check the anemometers.** Confirm that `PriAnemo` / `SecAnemo` correlate with `PrWindSp` / `SeWindSp` at ≈ 1, so that dropping them is safe.
4. **Merge June 2024 – June 2026** into two files: a turbine-level file (merged on `TimeStamp` + `StationId`) and a farm-level file (merged on `TimeStamp`). I will save them as Parquet, because there are about 2.2 million turbine rows.
5. **Store fractions, not decisions.** I will keep availability, curtailment and icing as fractions and flags instead of deleting rows. That way I can change the thresholds later without reprocessing the raw data.

****

****

In [17]:
KOLONE = {
    # --- farm level (merge on TimeStamp) ---
    "tblGrid": [
        "TimeStamp",
        "ActivePowerExport",   # TARGET candidate 1: export at the metering point
        "ActivePower",         # check against the sum of turbines
        "WPSStatus",           # wind farm status
    ],
    "tblGridScientific": [
        "TimeStamp",
        "ActiveEnergy",        # TARGET candidate 2: energy per 10 min
        "ActivePowerMean",
        "DataOk",              # quality flag of the grid measurement
    ],
    # --- turbine level (merge on TimeStamp + StationId) ---
    "tblSCTurGrid": [
        "TimeStamp", "StationId",
        "wtc_ActPower_mean",              # power per turbine
        "wtc_ActPower_min",               # ~0 while mean > 0 -> stop inside the interval
        "wtc_ActRegSt_endvalue",          # power regulation state
        "wtc_ActPower_Quality_endvalue",  # measurement quality
    ],
    "tblSCTurbine": [
        "TimeStamp", "StationId",
        "wtc_AcWindSp_mean", "wtc_AcWindSp_stddev",  # wind speed + turbulence
        "wtc_PrWindSp_mean", "wtc_SeWindSp_mean",    # anemometer check
        "wtc_ActualWindDirection_mean",              # wind direction
        "wtc_NacelPos_mean",                         # for the northing correction
        "wtc_PitcPosA_mean",                         # curtailment check (blades B, C move together)
        "wtc_PowerRef_endvalue",                     # power limit
        "wtc_BoostAct_mean",                         # power boost active
        "wtc_WTOperSt_endvalue",                     # operating state
    ],
    "tblSCTurTemp": [
        "TimeStamp", "StationId",
        "wtc_AmbieTmp_mean",                         # ambient temperature
    ],
    "tblSCTurFlag": [
        "TimeStamp", "StationId",
        "wtc_ScInOper_timeon",   # seconds in operation
        "wtc_ScReToOp_timeon",   # seconds ready, waiting for wind
        "wtc_ScTurSto_timeon",   # seconds stopped: turbine fault
        "wtc_ScGrdSto_timeon",   # seconds stopped: grid
        "wtc_ScComSto_timeon",   # seconds stopped: commanded (curtailment)
        "wtc_ScEnvSto_timeon",   # seconds stopped: environment (high wind, ice, cold)
    ],
    "tblSCTurDigiIn": [
        "TimeStamp", "StationId",
        "wtc_IceRelay_timeon",   # seconds with ice detected
        "wtc_PowerRed_timeon",   # seconds with power reduction active
    ],
}

In [22]:
DATA_ROOT = Path("../data")              # contains the folders 2024, 2025, 2026
OUT_DIR = Path("../data/SCADA/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = pd.period_range("2024-04", "2026-04", freq="M")
FARM_TABLES = ["tblGrid", "tblGridScientific"]
TURBINE_TABLES = [t for t in KOLONE if t not in FARM_TABLES]

ALL_FILES = {f.name: f for f in DATA_ROOT.rglob("*.csv")} 

def load_table(table, keys):
    """Read only the selected columns of one table, for all months of interest."""
    cols = KOLONE[table]
    parts, missing = [], []
    for m in MONTHS:
        name = f"{table}_{m.year}_{m.month:02d}.csv"
        if name in ALL_FILES:
            parts.append(pd.read_csv(ALL_FILES[name], usecols=lambda c: c in cols))
        else:
            missing.append(str(m))
    df = pd.concat(parts, ignore_index=True).drop_duplicates()
    df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], utc=True, format='ISO8601')
    floats = df.select_dtypes("float64").columns
    df[floats] = df[floats].astype("float32")                  
    print(f"{table}: {len(parts)} months, {len(df):,} rows, duplicate keys: {df.duplicated(keys).sum()}")
    if missing:
        print(f"   missing months: {', '.join(missing)}")
    return df

def merge_tables(tables, keys):
    dfs = [load_table(t, keys) for t in tables]
    merged = reduce(lambda a, b: a.merge(b, on=keys, how="outer"), dfs)
    return merged.sort_values(keys).reset_index(drop=True)

turbines = merge_tables(TURBINE_TABLES, ["TimeStamp", "StationId"])
farm = merge_tables(FARM_TABLES, ["TimeStamp"])

turbines.to_parquet(OUT_DIR / "scada_turbines_2024-04_2026-04.parquet", index=False)
farm.to_parquet(OUT_DIR / "scada_farm_2024-04_2026-04.parquet", index=False)
turbines.shape, farm.shape

tblSCTurGrid: 25 months, 2,249,810 rows, duplicate keys: 0
tblSCTurbine: 25 months, 2,249,828 rows, duplicate keys: 18
tblSCTurTemp: 25 months, 2,249,827 rows, duplicate keys: 17
tblSCTurFlag: 25 months, 2,249,815 rows, duplicate keys: 5
tblSCTurDigiIn: 25 months, 2,249,814 rows, duplicate keys: 4
tblGrid: 25 months, 109,441 rows, duplicate keys: 0
tblGridScientific: 25 months, 109,441 rows, duplicate keys: 0


((2249858, 25), (109441, 7))

In [19]:
t = pd.read_parquet("../data/SCADA/scada_turbines_2024-04_2026-04.parquet")
f = pd.read_parquet("../data/SCADA/scada_farm_2024-04_2026-04.parquet")

In [15]:
t.columns

Index(['TimeStamp', 'StationId', 'wtc_ActPower_min', 'wtc_ActPower_mean',
       'wtc_ActRegSt_endvalue', 'wtc_ActPower_Quality_endvalue',
       'wtc_PitcPosA_mean', 'wtc_PowerRef_endvalue', 'wtc_PrWindSp_mean',
       'wtc_SeWindSp_mean', 'wtc_AcWindSp_mean', 'wtc_AcWindSp_stddev',
       'wtc_NacelPos_mean', 'wtc_BoostAct_mean',
       'wtc_ActualWindDirection_mean', 'wtc_WTOperSt_endvalue',
       'wtc_AmbieTmp_mean', 'wtc_ScInOper_timeon', 'wtc_ScReToOp_timeon',
       'wtc_ScEnvSto_timeon', 'wtc_ScComSto_timeon', 'wtc_ScTurSto_timeon',
       'wtc_ScGrdSto_timeon', 'wtc_IceRelay_timeon', 'wtc_PowerRed_timeon'],
      dtype='str')

In [14]:
f.columns

Index(['TimeStamp', 'WPSStatus', 'ActivePower', 'ActivePowerExport', 'DataOk',
       'ActiveEnergy', 'ActivePowerMean'],
      dtype='str')

In [18]:
t.shape

(2249858, 25)

In [17]:
f.shape

(109441, 7)

In [16]:
(f.isna().mean() * 100).round(2).sort_values(ascending=False)

DataOk               100.0
WPSStatus              0.0
TimeStamp              0.0
ActivePower            0.0
ActivePowerExport      0.0
ActiveEnergy           0.0
ActivePowerMean        0.0
dtype: float64

In [10]:
(t.isna().mean() * 100).round(2).sort_values(ascending=False)

wtc_ActPower_Quality_endvalue    99.30
wtc_PrWindSp_mean                 0.01
wtc_ActPower_min                  0.01
wtc_PitcPosA_mean                 0.01
wtc_ActPower_mean                 0.01
wtc_ActualWindDirection_mean      0.01
wtc_AcWindSp_stddev               0.01
wtc_AcWindSp_mean                 0.01
wtc_SeWindSp_mean                 0.01
wtc_AmbieTmp_mean                 0.01
wtc_IceRelay_timeon               0.01
StationId                         0.00
TimeStamp                         0.00
wtc_PowerRef_endvalue             0.00
wtc_ActRegSt_endvalue             0.00
wtc_BoostAct_mean                 0.00
wtc_NacelPos_mean                 0.00
wtc_WTOperSt_endvalue             0.00
wtc_ScInOper_timeon               0.00
wtc_ScEnvSto_timeon               0.00
wtc_ScReToOp_timeon               0.00
wtc_ScComSto_timeon               0.00
wtc_ScTurSto_timeon               0.00
wtc_ScGrdSto_timeon               0.00
wtc_PowerRed_timeon               0.00
dtype: float64